# Extracting `Jx` during plasma oscillation

This notebook shows how to sample the current density `Jx` from the `electrostatic_1d` solver with `custom_field_loop`.

We use a 1D plasma oscillation as the example problem: a uniform electron plasma is placed in a periodic domain, the electric field is initialized with a sinusoidal perturbation, and after each `advance` step we read out the grid current with `field="Jx"`.

The key point is that `electrostatic_1d` exposes `Jx` directly through `custom_field_loop`, so the current can be copied into NumPy without reconstructing it from particle data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pipic
from numba import carray, cfunc
from pipic import consts, types

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

# --- Plasma oscillation setup -----------------------------------------------
density = 1e18
temperature = 1e-6 * consts.electron_mass * consts.light_velocity**2
nx = 128
x_min_factor = 64
debye_length = np.sqrt(temperature / (4 * np.pi * density * consts.electron_charge**2))
plasma_period = np.sqrt(np.pi * consts.electron_mass / (density * consts.electron_charge**2))
xmin, xmax = -x_min_factor * debye_length, x_min_factor * debye_length
time_step = plasma_period / 64
field_amplitude = 0.01 * 4 * np.pi * (xmax - xmin) * consts.electron_charge * density

sim = pipic.init(solver="electrostatic_1d", nx=nx, xmin=xmin, xmax=xmax)

@cfunc(types.add_particles)
def init_density(r, data_double, data_int):
    return density

sim.add_particles(
    name="electron",
    number=nx * 1000,
    charge=-consts.electron_charge,
    mass=consts.electron_mass,
    temperature=temperature,
    density=init_density.address,
 )

@cfunc(types.field_loop)
def set_field(ind, r, E, B, data_double, data_int):
    E[0] = field_amplitude * np.sin(2 * np.pi * r[0] / (xmax - xmin))

sim.field_loop(handler=set_field.address)

print("Particles:", sim.get_number_of_particles())

## 2. Sampling `Jx` with `custom_field_loop`

`custom_field_loop` is a field-solver callback. For `electrostatic_1d`, the implementation accepts the field names `"Ex"` and `"Jx"`.

Here we allocate a NumPy buffer, expose it to the callback with `pipic.addressof`, and copy `Jx` node by node into that buffer.

In [ ]:
Jx = np.zeros(nx, dtype=np.double)
node_x = np.linspace(xmin, xmax, nx, endpoint=False) + 0.5 * (xmax - xmin) / nx

@cfunc(types.custom_field_loop)
def get_Jx(it, r, field, data_double, data_int):
    out = carray(data_double, (nx,), dtype=np.double)
    out[it[0]] = field[0]


def sample_Jx():
    Jx.fill(0.0)
    sim.custom_field_loop(
        handler=get_Jx.address,
        field="Jx",
        data_double=pipic.addressof(Jx),
    )
    return Jx.copy()

## 3. Advance the simulation and plot `Jx`

The snippet below advances the plasma oscillation for a few time steps, samples `Jx` after each step, and plots the result as a time-space map.

If you want a lower-level diagnostic, you can also compute a single profile with `sample_Jx()` and inspect the NumPy array directly.

In [ ]:
steps = 80
Jx_history = np.zeros((steps, nx), dtype=np.double)

for i in range(steps):
    sim.advance(time_step=time_step, number_of_iterations=1)
    Jx_history[i, :] = sample_Jx()

fig, ax = plt.subplots(figsize=(10, 4))
image = ax.imshow(
    Jx_history,
    origin="lower",
    aspect="auto",
    extent=[xmin, xmax, 0, steps],
    cmap="RdBu_r",
)
ax.set_xlabel("x")
ax.set_ylabel("time step")
ax.set_title("Plasma oscillation: Jx sampled with custom_field_loop")
fig.colorbar(image, ax=ax, label="Jx")
plt.show()



## 4. What to change for your own diagnostic

To adapt this pattern to a different field diagnostic:

- keep the `@cfunc(types.custom_field_loop)` callback
- change `field="Jx"` to the field you want to inspect, when the solver supports it
- replace the NumPy output buffer with your own array or post-processing logic

For `electrostatic_1d`, `field="Jx"` is the direct way to read the grid current used by the solver.